# 13.2 语音语言 (Audio-Language)

语音语言模型将语音信号与文本理解结合，实现语音识别、语音合成等任务。

本节涵盖：语音编码器、语音语言模型、语音合成

## 1. 语音编码器与语音语言模型

**语音编码器**：将音频信号转换为特征表示。
- **Whisper**：OpenAI的鲁棒语音识别模型，680K小时多语言训练
- 流程：音频 -> 梅尔频谱图 -> Encoder-Decoder -> 文本

**语音语言模型**：将语音特征与LLM结合，支持语音输入理解和语音输出。
- **SpeechGPT**：语音理解 + 语音生成
- **Qwen-Audio**：音频理解 + 文本生成

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class SimpleAudioEncoder(nn.Module):
    def __init__(self, n_mels=64, d_model=128, n_heads=2, n_layers=1):
        super().__init__()
        self.mel_proj = nn.Linear(n_mels, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)

    def forward(self, mel_spec):
        h = self.mel_proj(mel_spec)
        return self.encoder(h)

class SpeechLM(nn.Module):
    def __init__(self, d_audio=128, d_text=128, vocab_size=500):
        super().__init__()
        self.audio_encoder = SimpleAudioEncoder()
        self.audio_proj = nn.Linear(d_audio, d_text)
        self.text_embed = nn.Embedding(vocab_size, d_text)
        self.decoder = nn.TransformerDecoderLayer(d_text, 2, d_text * 4, batch_first=True)
        self.head = nn.Linear(d_text, vocab_size)

    def forward(self, mel_spec, text_tokens):
        audio_features = self.audio_encoder(mel_spec)
        audio_tokens = self.audio_proj(audio_features)
        text_embeds = self.text_embed(text_tokens)
        h = self.decoder(text_embeds, audio_tokens)
        return self.head(h)

mel_spec = torch.randn(2, 100, 64)
text = torch.randint(0, 500, (2, 20))

audio_enc = SimpleAudioEncoder()
audio_features = audio_enc(mel_spec)

speech_lm = SpeechLM()
logits = speech_lm(mel_spec, text)

print('=== Audio-Language Model ===')
print(f'Mel spectrogram: {mel_spec.shape}')
print(f'Audio features: {audio_features.shape}')
print(f'Text tokens: {text.shape}')
print(f'Output logits: {logits.shape}')
print(f'\nKey: Audio encoder extracts features from mel spectrogram,')
print(f'then cross-attention fuses audio and text modalities.')

## 2. 语音合成 (TTS)

**目的**：将文本转换为自然语音

**现代方法**：
- **VITS**：端到端TTS，直接从文本生成波形
- **Bark/XTTS**：基于语言模型的TTS，生成语音token
- **Voicebox**：流匹配模型，支持语音编辑和风格迁移

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print('=== Text-to-Speech (TTS) ===')
print(f'\nModern TTS approaches:')
print(f'  1. VITS: End-to-end, text -> waveform directly')
print(f'  2. Bark/XTTS: LLM-based, generate speech tokens autoregressively')
print(f'  3. Voicebox: Flow matching, supports voice editing and style transfer')
print(f'\nKey insight: LLM-based TTS treats speech as just another language,')
print(f'enabling multilingual, multi-style generation with a single model.')